# Общая аналитическая зависимость по условиям эксперимента

Воспроизводимый пример второго этапа: `y(t,z) = f(t;g(z))` для экспоненциальной релаксации с наблюдаемым `y0`.

Данные генерируются в ноутбуке. Установка из корня проекта: `uv sync --extra dev --extra notebook`. Запуск: `uv run jupyter lab examples/multi_experiment.ipynb`.

Известные зависимости: `A(T)=0.2+0.015*T`, `k(T,pH)=exp(-3.2+0.04*T-0.15*(pH-7))`. Независимые эксперименты разделяются до подбора любых коэффициентов.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Math, display

from parametric_regression import ExponentialRelaxationFamily, MultiExperimentRegressor
from parametric_regression.multi_experiment.visualization import (
    plot_experiments,
    plot_new_trajectory,
    plot_parameter_dependency,
    plot_prediction_comparison,
)

rng = np.random.default_rng(42)
frames = []
for experiment in range(30):
    temperature = float(rng.uniform(18, 34))
    ph = float(rng.uniform(6, 8))
    time = np.linspace(0, 50, 50)
    initial = 0.9 + 0.01 * experiment
    amplitude = 0.2 + 0.015 * temperature
    rate = np.exp(-3.2 + 0.04 * temperature - 0.15 * (ph - 7))
    response = initial + amplitude * (-np.expm1(-rate * time))
    noise = rng.normal(0, 0.002, len(time))
    noise[0] = 0
    frames.append(
        pd.DataFrame(
            {
                "experiment_id": f"E{experiment:02d}",
                "time": time,
                "temperature": temperature,
                "pH": ph,
                "response": response + noise,
            }
        )
    )
df = pd.concat(frames, ignore_index=True)
display(df.head())
print("Независимых экспериментов:", df.experiment_id.nunique())

## Подбор по train и выбор по validation

Время в каждой траектории отсчитывается от начала. Первое измерение определяет `y0`, поэтому оно не входит в оценочную ошибку будущих наблюдений. Auto сравнивает Constant, Linear и quadratic для `A` и `log(k)` по RMSE итоговых кривых. Test используется только после выбора.

In [ ]:
model = MultiExperimentRegressor(
    ExponentialRelaxationFamily(),
    parameter_selection="auto",
    random_state=42,
)
result = model.fit(
    df,
    time_column="time",
    target_column="response",
    group_column="experiment_id",
    condition_columns=["temperature", "pH"],
)
assert result.success, result.message
print(result.message)
print("Диагностика:", result.warnings)
display(result.split_protocol.experiments)
display(result.experiment_parameters)
display(result.candidate_scores)
heldout = result.experiment_parameters.query("partition != 'train'")
assert heldout[["param:A", "param:k"]].isna().all().all()

In [ ]:
figure = plot_experiments(result, partition="train", max_trajectories=4)
plt.show()
plt.close(figure)

## Зависимости коэффициентов и общая формула

Точки показывают оценки train-траекторий при всех наблюдавшихся условиях. Линия на графике `k` — срез при `pH=7`; точки с другим pH не обязаны лежать на этом срезе. Формулы включают численные константы нормировки и используют исходные названия условий.

In [ ]:
print("Выбранные модели:", {name: r.model.name for name, r in result.parameter_models.items()})
for name, expression in result.parameter_formulas.items():
    print(f"{name} = {expression}")
print(result.formula)
display(Math(result.latex_formula))
figure = plot_parameter_dependency(result, "k", "temperature", reference_conditions={"pH": 7.0})
plt.show()
plt.close(figure)

## Новая комбинация условий

Начальное состояние передаётся отдельно. Прогноз сравнивается с известной порождающей зависимостью; для реального эксперимента такая истина обычно неизвестна.

In [ ]:
new_time = np.linspace(0, 50, 100)
new_conditions = {"temperature": 25.0, "pH": 7.0}
initial_value = 1.10
true_curve = initial_value + (0.2 + 0.015 * 25) * (-np.expm1(-np.exp(-3.2 + 0.04 * 25) * new_time))
prediction = result.predict(new_time, new_conditions, initial_value=initial_value)
print("Сценарий:", result.prediction_scenarios(new_conditions))
print("Диапазоны train:", result.condition_ranges)
print("RMSE относительно истинной кривой:", np.sqrt(np.mean((prediction - true_curve) ** 2)))
figure = plot_new_trajectory(
    result, new_time, new_conditions, initial_value=initial_value, truth=true_curve
)
plt.show()
plt.close(figure)
display(result.predict_parameters(new_conditions))

## Независимая оценка

Отдельно показываются индивидуальные подгонки train, общая модель на train, validation и test. При усреднении по экспериментам каждый независимый эксперимент имеет одинаковый вес. Ошибки коэффициентов не используются для выбора.

In [ ]:
summary = pd.DataFrame({name: metrics["experiment"] for name, metrics in result.metrics.items()}).T
display(summary)
display(pd.DataFrame({name: metrics["observation"] for name, metrics in result.metrics.items()}).T)
assert result.metrics["test"]["complete"]
figure = plot_prediction_comparison(result, partition="test")
plt.show()
plt.close(figure)

## Проверка промежуточных параметров на известной синтетике

Можно отдельно сравнить прогнозы коэффициентов с истинными значениями. Это диагностика уже выбранной модели и не участвует в обучении.

In [ ]:
known_conditions = pd.DataFrame({"temperature": [24.0, 25.0, 26.0], "pH": [6.8, 7.0, 7.2]})
true_parameters = pd.DataFrame(
    {
        "A": 0.2 + 0.015 * known_conditions.temperature,
        "k": np.exp(-3.2 + 0.04 * known_conditions.temperature - 0.15 * (known_conditions.pH - 7)),
    }
)
display(pd.DataFrame(result.score_parameters(known_conditions, true_parameters)).T)

## Ограничения

Условия постоянны внутри траектории, временное семейство задано явно. Нелинейный fitting локальный; ранг и обусловленность якобиана помогают исключать нестабильные оценки, но не заменяют анализ неопределённости. Начальное состояние здесь известно точно; шум в `y0` влияет на весь условный прогноз.

`interpolation` означает попадание в диапазоны каждого фактора, а не проверку выпуклой оболочки сочетаний. За пределами диапазонов выдаётся `ExtrapolationWarning`. Модель не переобучается по test и автоматически не объединяет train с validation.